In [ ]:
import pandas as pd

calendar = pd.read_csv('Data/calendar.csv')
items = pd.read_csv('Data/items_per_season.csv')
cost_amount = pd.read_csv('Data/cost_amount.csv')
mail_hashed_all = pd.read_csv('Data/mail_hashed_all.csv')
customers_fy_16_17 = pd.read_csv('Data/customers_fy_16_17.csv')
customers_fy_23_24 = pd.read_csv('Data/customers_fy_23_24.csv')
items_2016 = pd.read_csv('Data/items_2016.csv')
items_2022 = pd.read_csv('Data/items_2022.csv')
orders_fy_16_17 = pd.read_csv('Data/order_invoice_refunds_fy_16_17.csv')
orders_fy_22_23 = pd.read_csv('Data/order_invoice_refunds_fy_22_23.csv')

In [ ]:
display(calendar.head())


In [ ]:
# hier stehen die echten Spaltennamen in Row 0
items = pd.read_csv("Data/items_per_season.csv", skiprows=1, names=["ITEM_NO", "SEASON", "IS_CURRENT_SEASON", "ITEM_END_DATE", "SEASON_START_DATE", "SEASON_END_DATE", "PRODUCT_STATUS_PER_SEASON", "IS_ACTIVE_IN_SEASON"])
items = items.drop(index=0).reset_index(drop=True)
display(items.head())

In [ ]:
display(cost_amount.head())

In [ ]:
display(mail_hashed_all.head())

In [ ]:

display(customers_fy_16_17.head())


In [ ]:
display(customers_fy_23_24.head())


In [ ]:
display(items_2016.head())


In [ ]:
display(items_2022.head())


In [ ]:
display(orders_fy_16_17.head())


In [ ]:
display(orders_fy_22_23.head())

# Basic merging & dropping duplicates
- Merge both dfs of customers, items, orders
- add mail_hashed_all to customers
- add cost_amount to orders
- Drop exact duplicates (rows where all values are similar)
Goal: Get a clean data baseline

In [ ]:
# Merge all data of customers, items, orders, and add hashed mails to customers
customers_df = pd.concat([customers_fy_16_17, customers_fy_23_24], ignore_index=True)
orders_df = pd.concat([orders_fy_16_17, orders_fy_22_23], ignore_index=True)
items_df = pd.concat([items_2016, items_2022], ignore_index=True)

In [ ]:
# Check for exact duplicates
print("Number of duplicate rows in customers_df:", customers_df.duplicated().sum())
print("Number of duplicate rows in orders_df:", orders_df.duplicated().sum())
print("Number of duplicate rows in items_df:", items_df.duplicated().sum())

In [ ]:
# Drop exact duplicates if any
customers_dedup = customers_df.drop_duplicates().reset_index(drop=True)
orders_dedup = orders_df.drop_duplicates().reset_index(drop=True)
items_dedup = items_df.drop_duplicates().reset_index(drop=True)

# Check for exact duplicates again after dropping
print("Number of duplicate rows in customers_dedup:", customers_dedup.duplicated().sum())
print("Number of duplicate rows in orders_dedup:", orders_dedup.duplicated().sum())
print("Number of duplicate rows in items_dedup:", items_dedup.duplicated().sum())

In [ ]:
# Add hashed mails to customers
customers_all = customers_dedup.merge(mail_hashed_all, on='CUSTOMER_NO', how='left')
display(customers_all.head())

In [ ]:
print("Number of duplicated Customer IDs in customers_all:", customers_all['CUSTOMER_NO'].duplicated().sum())
print("Number of duplicated hashed mails in customers_all:", customers_all['EMAIL_HASHED'].duplicated().sum())
print("Number of duplicated combos of Customer ID and hashed mail in customers_all:", customers_all.duplicated(subset=['CUSTOMER_NO', 'EMAIL_HASHED']).sum())

In [ ]:
# Check for multiple emails per customer and multiple customers per email
print(f"Found {(customers_all.groupby('CUSTOMER_NO')['EMAIL_HASHED'].nunique() > 1).sum()} Customer IDs with multiple emails.")
print(f"Found {(customers_all.groupby('EMAIL_HASHED')['CUSTOMER_NO'].nunique() > 1).sum()} Emails tied to multiple Customer IDs.")

# --> we can use the hashed mails as unique identifiers for the customers

In [ ]:
# Add the cost amount to orders
print("Number of duplicated rows in orders_dedup based on DOCUMENT_NO:", orders_dedup['DOCUMENT_NO'].duplicated().sum())
print("Number of duplicated rows in orders_dedup based on DOCUMENT_NO and LINE_NO:", orders_dedup.duplicated(subset=['DOCUMENT_NO', 'LINE_NO']).sum())
orders_all = orders_dedup.merge(cost_amount, on=['DOCUMENT_NO', 'LINE_NO'], how='left')
display(orders_all.head())

In [ ]:
# 1. Orders kombinieren & auf Order-Ebene aggregieren 
orders_all = pd.concat([orders_fy_16_17, orders_fy_22_23], ignore_index=True)
orders_all['ORDER_DATE'] = pd.to_datetime(orders_all['ORDER_DATE'], errors='coerce')

order_level = (
    orders_all
    .groupby(['CUSTOMER_NO', 'ORDER_NO', 'ORDER_DATE'])
    .agg(
        net_amount_order = ('NET_AMOUNT',           'sum'),
        qty_order        = ('QUANTITY',              'sum'),
        n_items          = ('ITEM_NO',               'nunique'),
        has_discount     = ('LINE_DISCOUNT_AMOUNT',  'any'),
        discount_code    = ('DISCOUNT_CODE',         'first'),  
    )
    .reset_index()
)
order_level['is_return_order'] = order_level['qty_order'] <= 0



In [ ]:
display(orders_all.head())

In [ ]:
print(customers_fy_23_24['CUSTOMER_NO'].duplicated().sum(), "Duplikate")
print(f"Gesamt: {len(customers_fy_23_24)} Zeilen | Unique: {customers_fy_23_24['CUSTOMER_NO'].nunique()} Kunden")


In [ ]:
dupes = customers_fy_23_24[customers_fy_23_24['CUSTOMER_NO'].duplicated(keep=False)]

# Welche Spalten variieren innerhalb derselben CUSTOMER_NO?
varying_cols = (
    dupes.groupby('CUSTOMER_NO')
    .nunique()
    .gt(1)          # True wo Werte unterschiedlich sind
    .any()          # Spalte hat mind. 1 Kunde mit Variation
)
print("Spalten die variieren:")
print(varying_cols[varying_cols].index.tolist())

In [ ]:
# 2. Kunden (customers hat mehrere Zeilen pro Kunde, wo sonst alles gleich ist?) 
customers_dedup = customers_fy_23_24.drop_duplicates(subset='CUSTOMER_NO')



In [ ]:
# 3. Panel: Inner Join  nur Kunden die in customers_fy_23_24 sind 
panel = (
    order_level
    .merge(
        customers_dedup[['CUSTOMER_NO', 'CUSTOMER_TYPE', 'COUNTRY_REGION_CODE',
                         'CUSTOMER_ORDER_TYPE', 'HAS_CUSTOMER_ACCOUNT', 'IS_SUBSCRIBED_FOR']],
        on='CUSTOMER_NO',
        how='inner'
    )
    .sort_values(['CUSTOMER_NO', 'ORDER_DATE'])
    .reset_index(drop=True)
)

# Chronologische Bestellnummer pro Kunde (1 = erste Bestellung)
panel['order_rank'] = (
    panel.groupby('CUSTOMER_NO')['ORDER_DATE']
    .rank(method='dense').astype(int)
)

print(f"Panel: {panel['CUSTOMER_NO'].nunique():,} Kunden | {len(panel):,} Bestellungen")
print(f"Davon Retouren (komplett): {panel['is_return_order'].sum():,}")
display(panel[['CUSTOMER_NO','order_rank','ORDER_DATE','ORDER_NO',
               'net_amount_order','qty_order','is_return_order','CUSTOMER_ORDER_TYPE']].head(15))

In [ ]:
display(panel.head())

In [ ]:
# Items: beide Jahrgänge kombinieren, dedup auf ITEM_NO
items_all = (
    pd.concat([items_2016, items_2022], ignore_index=True)
    .drop_duplicates(subset='ITEM_NO')
)

# Merge auf orders_all (item-level, vor Order-Aggregation)
orders_enriched = orders_all.merge(
    items_all[['ITEM_NO', 'BRAND', 'MAIN_CATEGORY', 'SUB_CATEGORY', 'PRODUCT_TYPE', 'SIZE']],
    on='ITEM_NO',
    how='left'
)


In [ ]:

orders_enriched['is_return'] = orders_enriched['QUANTITY'] < 0

footwear_per_order = (
    orders_enriched[
        (orders_enriched['MAIN_CATEGORY'] == 'FOOTWEAR') &
        (~orders_enriched['is_return'])
    ]
    .groupby(['CUSTOMER_NO', 'ORDER_NO'])['SIZE']
    .apply(lambda x: sorted(pd.to_numeric(x, errors='coerce').dropna().unique().tolist()))
    .reset_index(name='footwear_sizes')
)

panel = panel.merge(footwear_per_order, on=['CUSTOMER_NO', 'ORDER_NO'], how='left')

print(f"Kunden mit mind. einer Schuhgröße: {panel['footwear_sizes'].notna().sum():,} Bestellungen")
display(panel[panel['footwear_sizes'].notna()][['CUSTOMER_NO','ORDER_DATE','footwear_sizes']].head(10))

In [ ]:
import matplotlib.pyplot as plt

# Anzahl verschiedener Schuhgrößen pro Bestellung
panel['n_shoe_sizes'] = panel['footwear_sizes'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# Kategorien: 0, 1, 2, 3+
bins = panel['n_shoe_sizes'].clip(upper=3).value_counts().sort_index()
bins.index = ['0 Größen', '1 Größe', '2 Größen', '3+ Größen']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(bins.index, bins.values, color='steelblue', edgecolor='white')

for bar, val in zip(bars, bins.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=10)

ax.set_title('Anzahl verschiedener Schuhgrößen pro Bestellung')
ax.set_xlabel('Anzahl Schuhgrößen')
ax.set_ylabel('Anzahl Bestellungen')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print(bins.to_string())


In [ ]:
# Alle Schuhgrößen pro Kunde über alle Bestellungen aggregieren
sizes_per_customer = (
    panel[panel['footwear_sizes'].apply(lambda x: isinstance(x, list) and len(x) > 0)]
    .groupby('CUSTOMER_NO')['footwear_sizes']
    .apply(lambda x: len(set(s for sizes in x for s in sizes)))
    .reset_index(name='n_unique_sizes')
)

# Kunden ohne Footwear = 0
all_customers = panel[['CUSTOMER_NO']].drop_duplicates()
sizes_per_customer = all_customers.merge(sizes_per_customer, on='CUSTOMER_NO', how='left')
sizes_per_customer['n_unique_sizes'] = sizes_per_customer['n_unique_sizes'].fillna(0).astype(int)

# Kategorien
bins = sizes_per_customer['n_unique_sizes'].clip(upper=3).value_counts().sort_index()
bins.index = ['0 Größen', '1 Größe', '2 Größen', '3+ Größen']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(bins.index, bins.values, color='steelblue', edgecolor='white')

for bar, val in zip(bars, bins.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontsize=10)

ax.set_title('Anzahl verschiedener Schuhgrößen pro Kunde (kumuliert über alle Bestellungen)')
ax.set_xlabel('Anzahl unique Schuhgrößen')
ax.set_ylabel('Anzahl Kunden')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print(bins.to_string())
